# 02 — Skeleton Extraction: ShuttleSet (Fast — YOLO-only)

Extracts dual-player skeletons for ShuttleSet using **YOLOv8-Pose only** (no Grounding DINO).
Player filtering uses the existing spatial-fraction thresholds that are tuned for broadcast-angle footage.

**Speed optimisations over the GDINO version:**
1. **No Grounding DINO** — removes the heavy VL-transformer (~50% of per-frame time)
2. **Batched YOLO inference** — GPU processes 16 frames at once instead of 1
3. **Local-SSD I/O** — frames copied from Drive to `/content/` before processing
4. **FP16 inference** — half-precision on GPU for ~1.5x throughput
5. **Dual-model support** — `yolov8s-pose` (default) + `yolov8n-pose` (fast) selectable per match

**Output format:** `(2, T, 34)` per-match `.npy` — identical to GDINO version
- `dim 0` = [x, y]
- `dim 1` = T frames
- `dim 2` = 34 joints: nodes 0–16 = P0 (top-court / smaller Y), 17–33 = P1 (bottom-court)

**Saved to:** `datasets_preprocessing/shuttleset_skeletons_fast/{match}/skeletons.npy`

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

In [ ]:
! pip install ultralytics

In [ ]:
import os, json, cv2, shutil, time
import numpy as np
from pathlib import Path
from tqdm import tqdm
from collections import defaultdict
import torch

# === PATHS ===
DRIVE_DIR    = Path('/content/drive/MyDrive/Baddiev2')
SS_FRAMES    = DRIVE_DIR / 'datasets_preprocessing/shuttleset_frames'
SS_OUTPUTS   = DRIVE_DIR / 'datasets_preprocessing/shuttleset_outputs'
SS_SKELETONS = DRIVE_DIR / 'datasets_preprocessing/shuttleset_skeletons_fast'   # NEW output dir
FAILED_LOG   = DRIVE_DIR / 'datasets_preprocessing/failed_skeletons_fast.json'

SS_SKELETONS.mkdir(parents=True, exist_ok=True)

failed_matches = json.loads(FAILED_LOG.read_text()) if FAILED_LOG.exists() else []

print(f"Frames dir:    {SS_FRAMES}")
print(f"Skeletons dir: {SS_SKELETONS}")
print(f"Failed matches: {len(failed_matches)}")

In [ ]:
from ultralytics import YOLO

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")

# Load both model sizes
print("Loading YOLOv8s-pose (default)...")
yolo_s = YOLO('yolov8s-pose.pt')

print("Loading YOLOv8n-pose (fast)...")
yolo_n = YOLO('yolov8n-pose.pt')

# Enable FP16 on GPU
if DEVICE == 'cuda':
    yolo_s.model.half()
    yolo_n.model.half()
    print("  FP16 enabled for both models")

print("Models ready.")

## Spatial Filter + Extraction Helpers

Same spatial-fraction thresholds from the GDINO version. These are tuned for
broadcast-angle badminton footage and reliably reject chair umpires, ball kids,
and spectators without needing GDINO's semantic detection.

In [ ]:
# -- Spatial filter thresholds (broadcast-angle assumption) ---------------------
# Chair umpire sits at x ~ 5-10% of width on the side. Players are always in
# the middle 75% of the frame. Tiny boxes (< 10% frame height) = ball kids /
# distant spectators.
_X_MIN_FRAC   = 0.234   # was 0.12
_X_MAX_FRAC   = 0.755   # was 0.88
_Y_MIN_FRAC   = 0.185   # was 0.05
_MIN_H_FRAC   = 0.10    # minimum box height as fraction of frame height


def _in_court_region(box, W, H):
    """True if box center is in the valid court zone and the box is large enough."""
    cx = (box[0] + box[2]) / 2
    cy = (box[1] + box[3]) / 2
    box_h = box[3] - box[1]
    return (W * _X_MIN_FRAC <= cx <= W * _X_MAX_FRAC and
            cy >= H * _Y_MIN_FRAC and
            box_h >= H * _MIN_H_FRAC)


# -- Per-frame extraction (from batched YOLO results) -------------------------

def _extract_frame(res, H, W):
    """Extract 2-player skeleton from a single YOLO result object."""
    out = np.zeros((2, 34), dtype=np.float32)

    if res.keypoints is None or res.boxes is None:
        return out

    y_boxes = res.boxes.xyxy.cpu().numpy()
    y_cls   = res.boxes.cls.cpu().numpy()
    y_kpts  = res.keypoints.xy.cpu().numpy()     # (N, 17, 2)
    y_confs = res.boxes.conf.cpu().numpy()

    # Filter: person class + in court region
    valid = [
        i for i, (yb, yc) in enumerate(zip(y_boxes, y_cls))
        if int(yc) == 0 and _in_court_region(yb, W, H)
    ]

    if len(valid) < 2:
        return out

    # Top-2 by confidence
    valid.sort(key=lambda i: -y_confs[i])
    top2 = valid[:2]
    kpts_kept = [y_kpts[j] for j in top2]
    cys  = [float(y_kpts[j][:, 1].mean()) for j in top2]

    # Y-sort: P0 = top-court (smaller Y), P1 = bottom-court
    ys = np.argsort(cys)
    p0, p1 = kpts_kept[ys[0]], kpts_kept[ys[1]]
    out[0, :17] = p0[:, 0];  out[1, :17] = p0[:, 1]
    out[0, 17:] = p1[:, 0];  out[1, 17:] = p1[:, 1]
    return out


print("Spatial filter + extraction helpers ready.")

In [ ]:
# === CONFIG ===
BATCH_SIZE     = 3           # matches per run
FRAME_BATCH    = 16          # frames per YOLO batch (GPU saturation)
USE_NANO       = False       # True = yolov8n-pose for ALL matches

# Override: use nano for specific matches only (set USE_NANO=False to enable this)
NANO_MATCHES = {
    # 'some_easy_match_name',
}

# Target specific matches (empty = auto-queue all pending)
TARGET_MATCH = [
#    'Anders_Antonsen_Sameer_Verma_TOYOTA_THAILAND_OPEN_2021_QuarterFinals',
#    'CHOU_Tien_Chen_Jonatan_CHRISTIE_Indonesia_Open_2019_Quarter-finals',
#    'Hans-Kristian_Solberg_Vittinghus_Anders_Antonsen_TOYOTA_THAILAND_OPEN_2021_SemiFinals',
    'Hans-Kristian_Solberg_Vittinghus_Lee_Cheuk_Yu_TOYOTA_THAILAND_OPEN_2021_QuarterFinals',
   'NG_Ka_Long_Angus_Jonatan_CHRISTIE_Malaysia_Masters_2020_QuarterFinals',
   'Ng_Ka_Long_Angus_Kidambi_Srikanth_HSBC_BWF_WORLD_TOUR_FINALS_2020_QuarterFinals'
#    'Viktor_Axelsen_Anthony_Sinisuka_Ginting_YONEX_Thailand_Open_2021_SemiFinals',
#    'Viktor_Axelsen_Hans-Kristian_Solberg_VIittinghus_TOYOTA_THAILAND_OPEN_2021_Finals',
#    'Viktor_Axelsen_Jonatan_Christie_YONEX_Thailand_Open_2021_QuarterFinals',
#    'Viktor_Axelsen_Liew_Daren_TOYOTA_THAILAND_OPEN_2021_QuarterFinals',
#    'Viktor_Axelsen_Ng_Ka_Long_Angus_YONEX_Thailand_Open_2021_Finals',
]

# === AUDIT ===
source_matches    = sorted([d.name for d in SS_FRAMES.iterdir() if d.is_dir()])
processed_matches = sorted([d.name for d in SS_SKELETONS.iterdir()
                            if d.is_dir() and any(d.glob('*.npy'))])

if TARGET_MATCH:
    pending_matches = [m for m in TARGET_MATCH if (SS_FRAMES / m).exists()]
    missing = [m for m in TARGET_MATCH if m not in pending_matches]
    if missing:
        print(f"\u274c Not found: {missing}")
    print(f"\U0001f3af Target matches selected: {len(pending_matches)}")
else:
    pending_matches = [m for m in source_matches
                       if m not in processed_matches and m not in failed_matches]

print(f"Source: {len(source_matches)} | Done: {len(processed_matches)} "
      f"| Failed: {len(failed_matches)} | Pending: {len(pending_matches)}")
print(f"\nNext {min(BATCH_SIZE, len(pending_matches))} to process:")
for m in pending_matches[:BATCH_SIZE]:
    n_frames = len(list((SS_FRAMES / m).glob('*.jpg')))
    print(f"  \u2022 {m} ({n_frames} frames)")

In [ ]:
# === BATCH PROCESS ===
batch = pending_matches[:BATCH_SIZE]
success_count = 0

for batch_idx, match_name in enumerate(batch, 1):
    print(f"\n[{batch_idx}/{len(batch)}] {match_name}")
    print("-" * 60)

    match_frames_dir = SS_FRAMES / match_name
    match_skel_dir   = SS_SKELETONS / match_name

    # Skip if already done
    if match_skel_dir.exists() and any(match_skel_dir.glob('*.npy')):
        print("  SKIP: Already processed")
        continue

    # -- 1. Copy frames to local SSD ------------------------------------------
    local_dir = Path(f'/content/local_frames/{match_name}')
    if not local_dir.exists():
        print("  Copying frames to local SSD...")
        t0 = time.time()
        shutil.copytree(match_frames_dir, local_dir)
        print(f"  Copied in {time.time()-t0:.1f}s")

    frame_files = sorted(local_dir.glob('*.jpg'),
                         key=lambda p: int(p.stem.replace('frame_', '')))

    if not frame_files:
        print("  SKIP: No frames found")
        failed_matches.append(match_name)
        FAILED_LOG.write_text(json.dumps(list(set(failed_matches)), indent=2))
        shutil.rmtree(local_dir, ignore_errors=True)
        continue

    print(f"  Frames: {len(frame_files)}")

    # -- 2. Select model -------------------------------------------------------
    if USE_NANO or match_name in NANO_MATCHES:
        model = yolo_n
        model_tag = 'nano'
    else:
        model = yolo_s
        model_tag = 'small'
    print(f"  Model: yolov8{model_tag[0]}-pose")

    # -- 3. Batched extraction -------------------------------------------------
    match_skel_dir.mkdir(parents=True, exist_ok=True)
    all_skeletons = []
    frame_nums    = []
    t0 = time.time()

    try:
        for i in tqdm(range(0, len(frame_files), FRAME_BATCH),
                      desc="  Extracting", leave=False):
            chunk_files = frame_files[i : i + FRAME_BATCH]

            # Read batch of images
            imgs = [cv2.imread(str(f)) for f in chunk_files]

            # Batched YOLO inference (fp16 already set on model)
            results = model(imgs, verbose=False)

            for j, (fp, res) in enumerate(zip(chunk_files, results)):
                fnum = int(fp.stem.replace('frame_', ''))
                frame_nums.append(fnum)
                fH, fW = imgs[j].shape[:2]
                skel = _extract_frame(res, fH, fW)
                all_skeletons.append(skel)

        # Stack into (2, T, 34) array
        stacked = np.stack(all_skeletons, axis=1)
        elapsed = time.time() - t0
        fps_val = len(frame_files) / elapsed

        # Save
        np.save(match_skel_dir / 'skeletons.npy', stacked)
        np.save(match_skel_dir / 'frame_nums.npy', np.array(frame_nums))

        print(f"  \u2713 Saved: {stacked.shape}  ({elapsed:.0f}s, {fps_val:.1f} fps)")
        success_count += 1

    except Exception as e:
        print(f"  \u2717 Failed: {e}")
        import traceback; traceback.print_exc()
        failed_matches.append(match_name)
        FAILED_LOG.write_text(json.dumps(list(set(failed_matches)), indent=2))

    # Cleanup local copy
    shutil.rmtree(local_dir, ignore_errors=True)

# Summary
remaining = len(pending_matches) - len(batch)
print(f"\n{'='*60}")
print(f"DONE: {success_count}/{len(batch)} succeeded | {remaining} remaining")

In [ ]:
# === OPTIONAL: View/clear failed matches ===
print("Failed matches:")
for m in failed_matches:
    print(f"  \u2022 {m}")

# Uncomment to clear and retry:
#failed_matches = []
#FAILED_LOG.write_text('[]')
#print("Cleared")

In [ ]:
# === VERIFICATION ===
skel_files = list(SS_SKELETONS.rglob('skeletons.npy'))
print(f"Total skeleton files: {len(skel_files)}")

if skel_files:
    sample = np.load(skel_files[0])
    print(f"Sample shape: {sample.shape}")
    print(f"Sample match: {skel_files[0].parent.name}")